# Synthetic Data Generator
### Step 6.5: Kafka Producer Queue Manager Testing

In [5]:
import os

from utils.database.postgres import (
    get_engine_from_env,
    read_sql_dataframe,
    execute_sql,
    sanitize_sql_identifier,
)

from utils.synthetic.pipeline.producer_queue_manager import (
    ensure_send_queue_runtime_columns,
    ensure_simulation_state_control_table,
    upsert_simulation_state_control,
    read_simulation_state_control,
    claim_pending_send_queue_batch,
    claim_pending_sensor_messages_batch,
    mark_claimed_batch_sent, 
    mark_claimed_batch_failed,
    requeue_failed_messages,
    release_stale_claims,
    get_send_queue_status_counts
)



In [ ]:
SCHEMA = os.getenv("CAPSTONE_SCHEMA", "capstone")

DATASET_ID = os.getenv("SYNTHETIC_DATASET_ID", "pump_synthetic_v1")
RUN_ID = os.getenv("SYNTHETIC_RUN_ID", "synthetic_run_001")

SIMULATION_TABLE_NAME = str("simulation_state_control")
SEND_QUEUE_TABLE_NAME = "synthetic_sensor_messages_send_queue"

PRODUCER_TOPIC = os.getenv(
    "SYNTHETIC_KAFKA_TOPIC",
    "pump.telemetry.synthetic",
)

PRODUCER_WORKER_ID = os.getenv(
    "SYNTHETIC_PRODUCER_WORKER_ID",
    "producer_worker_test_001",
)


# -----------------------------------------------------------------------------
# Derived batch sizing for validation / producer pickup preview
# -----------------------------------------------------------------------------
NUMBER_OF_SENSORS = int(
    read_sql_dataframe(
        engine,
        f"""
        SELECT COUNT(DISTINCT sensor_index) AS sensor_count
        FROM "{SCHEMA}"."{SEND_QUEUE_SOURCE_TABLE_NAME}"
        """
    ).loc[0, "sensor_count"]
)

OBSERVATION_BATCH_SIZE = int(os.getenv("SYNTHETIC_OBSERVATION_BATCH_SIZE", "500"))
PRODUCER_BATCH_SIZE = OBSERVATION_BATCH_SIZE * NUMBER_OF_SENSORS
PRODUCER_POLL_SECONDS = float(os.getenv("SYNTHETIC_PRODUCER_POLL_SECONDS", "0.0"))
PRODUCER_MAX_SEND_ATTEMPTS = int(os.getenv("SYNTHETIC_PRODUCER_MAX_SEND_ATTEMPTS", "3"))

IS_ENABLED_FLAG = True
CLAIM_BATCH_SIZE = int(500)
STALE_CLAIM_RELEASE_MINUTES = int(15)

CONTROL_OWNER_ROLE = str("kafka_producer")
APPLY_OWNER_AND_GRANTS_FLAG = bool(False)


print("Producer queue manager test config")
print("schema:", SCHEMA)
print("send queue table:", SEND_QUEUE_TABLE_NAME)
print("producer topic:", PRODUCER_TOPIC)
print("producer worker id:", PRODUCER_WORKER_ID)
print("number of sensors:", NUMBER_OF_SENSORS)
print("observation batch size:", OBSERVATION_BATCH_SIZE)
print("message batch size:", PRODUCER_BATCH_SIZE)
print("apply owner/grants:", APPLY_OWNER_AND_GRANTS_FLAG)

Producer queue manager test config
schema: capstone
send queue table: synthetic_sensor_messages_send_queue
producer topic: pump.telemetry.synthetic
producer worker id: producer_worker_test_001
number of sensors: 52
observation batch size: 500
message batch size: 26000


---

In [7]:

engine = get_engine_from_env()


---

In [ ]:
# Optional run-control context check.
# The claim/reset test below does not depend on this row unless we explicitly
# choose to block claims based on simulation state.

control_row = read_simulation_state_control(
    engine=engine,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    schema=SCHEMA,
    table_name=SIMULATION_TABLE_NAME,
    owner_role=CONTROL_OWNER_ROLE,
    apply_owner_and_grants=APPLY_OWNER_AND_GRANTS_FLAG,
)

display(control_row)

---

In [ ]:
queue_health_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        queue_status,
        COUNT(*) AS row_count,
        COUNT(*) FILTER (WHERE claim_token IS NULL) AS unclaimed_count,
        COUNT(*) FILTER (WHERE claim_token IS NOT NULL) AS claimed_count,
        COUNT(*) FILTER (WHERE producer_sent_at IS NULL) AS unsent_count,
        COUNT(*) FILTER (WHERE producer_sent_at IS NOT NULL) AS sent_timestamp_count
    FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
    GROUP BY queue_status
    ORDER BY row_count DESC
    """
)

display(queue_health_dataframe)

,queue_status,row_count,unclaimed_count,claimed_count,unsent_count,sent_timestamp_count
0,pending,11700000,11700000,0,11700000,0


---

In [ ]:
producer_pickup_explain_dataframe = read_sql_dataframe(
    engine,
    f"""
    EXPLAIN ANALYZE
    SELECT
        message_key,
        observation_index,
        message_sequence_index,
        sensor_index
    FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
    WHERE queue_status = 'pending'
    ORDER BY observation_index, message_sequence_index, sensor_index
    LIMIT 26000
    """
)

display(producer_pickup_explain_dataframe)

,QUERY PLAN
0,Limit (cost=0.56..4286.20 rows=26000 width=76...
1,-> Index Scan using idx_synthetic_sensor_me...
2,Filter: (queue_status = 'pending'::text)
3,Planning Time: 0.323 ms
4,Execution Time: 24.929 ms


---

In [ ]:
claim_token, claimed_dataframe = claim_pending_sensor_messages_batch(
    engine=engine,
    schema=SCHEMA,
    queue_table=SEND_QUEUE_TABLE_NAME,
    batch_size=PRODUCER_BATCH_SIZE,
    producer_worker_id=PRODUCER_WORKER_ID,
    producer_topic=PRODUCER_TOPIC,
)

print("Claim token:", claim_token)
print("Claimed row count:", len(claimed_dataframe))
print("Expected claimed row count:", PRODUCER_BATCH_SIZE)

display(claimed_dataframe.head(104))

In [ ]:
'''
claimed_dataframe = claim_pending_send_queue_batch(
    engine=engine,
    dataset_id=DATASET_ID,
    run_id = RUN_ID,
    batch_size=CLAIM_BATCH_SIZE,
    schema=SCHEMA,
    table_name=SEND_QUEUE_TABLE_NAME,
    producer_topic=PRODUCER_TOPIC,
    producer_worker_id=PRODUCER_WORKER_ID,
)

print("Claimed rows:", len(claimed_dataframe))
'''

---

In [ ]:
claim_validation_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        queue_status,
        COUNT(*) AS row_count
    FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
    GROUP BY queue_status
    ORDER BY row_count DESC
    """
)

display(claim_validation_dataframe)

---

In [ ]:
claim_status_validation_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        queue_status,
        COUNT(*) AS row_count,
        COUNT(*) FILTER (WHERE claim_token IS NULL) AS null_claim_token_count,
        COUNT(*) FILTER (WHERE claim_token IS NOT NULL) AS populated_claim_token_count,
        COUNT(*) FILTER (WHERE claimed_at IS NULL) AS null_claimed_at_count,
        COUNT(*) FILTER (WHERE claimed_at IS NOT NULL) AS populated_claimed_at_count,
        COUNT(*) FILTER (WHERE producer_worker_id IS NOT NULL) AS populated_worker_count,
        COUNT(*) FILTER (WHERE producer_topic IS NOT NULL) AS populated_topic_count
    FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
    GROUP BY queue_status
    ORDER BY row_count DESC
    """
)

display(claim_status_validation_dataframe)

----

In [ ]:
display(
    claimed_dataframe[
        [
            "claim_token",
            "observation_index",
            "message_sequence_index",
            "sensor_name",
            "sensor_index",
            "queue_status",
            "producer_topic",
            "producer_worker_id",
        ]
    ].head(20)
)

---

In [ ]:
if not claimed_dataframe.empty:
    claim_token = str(claimed_dataframe["claim_token"].iloc[0])

    sent_dataframe = mark_claimed_batch_sent(
        engine=engine,
        claim_token=claim_token,
        schema=SCHEMA,
        table_name=SEND_QUEUE_TABLE_NAME,
    )

    print("Marked sent:", len(sent_dataframe))
    display(sent_dataframe)
else:
    print("No claimed rows to mark as sent.")

In [ ]:
if not claimed_dataframe.empty:
    claim_token = str(claimed_dataframe["claim_token"].iloc[0])

    failed_dataframe = mark_claimed_batch_failed(
        engine=engine,
        claim_token=claim_token,
        error_message="Kafka publish failed during test run.",
        schema=SCHEMA,
        table_name=SEND_QUEUE_TABLE_NAME,
    )

    print("Marked failed:", len(failed_dataframe))
    display(failed_dataframe.head())
else:
    print("No claimed rows to mark as failed.")

---

In [ ]:
requeued_dataframe = requeue_failed_messages(
    engine=engine,
    max_send_attempts = PRODUCER_MAX_SEND_ATTEMPTS,
    schema=SCHEMA,
    table_name=SEND_QUEUE_TABLE_NAME,
)

print("Requeued failed rows:", len(requeued_dataframe))
display(requeued_dataframe.head())

---

In [ ]:
released_dataframe = release_stale_claims(
    engine=engine,
    stale_after_minutes=STALE_CLAIM_RELEASE_MINUTES,
    max_send_attempts=PRODUCER_MAX_SEND_ATTEMPTS,
    schema=SCHEMA,
    table_name=SEND_QUEUE_TABLE_NAME,
)

print("Released stale claims:", len(released_dataframe))
display(released_dataframe.head())

---

In [ ]:
status_dataframe = get_send_queue_status_counts(
    engine=engine,
    schema=SCHEMA,
    table_name=SEND_QUEUE_TABLE_NAME,
)

display(status_dataframe)

---

In [ ]:
'''
RESET_TEST_CLAIMS_FLAG = True

if RESET_TEST_CLAIMS_FLAG:
    reset_dataframe = read_sql_dataframe(
        engine,
        f"""
        UPDATE "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
        SET
            queue_status = 'pending',
            claim_token = NULL,
            claimed_at = NULL,
            producer_topic = NULL,
            producer_worker_id = NULL
        WHERE queue_status = 'claimed'
          AND producer_sent_at IS NULL
        RETURNING COUNT(*) AS reset_count
        """
    )

    display(reset_dataframe)
'''

----

In [ ]:
# This is a safter two-step method, prevents errors like: Postgres complaining about RETURNING COUNT(*)
RESET_TEST_CLAIMS_FLAG = True

if RESET_TEST_CLAIMS_FLAG:
    before_reset_dataframe = read_sql_dataframe(
        engine,
        f"""
        SELECT COUNT(*) AS reset_candidate_count
        FROM "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
        WHERE queue_status = 'claimed'
          AND producer_sent_at IS NULL
        """
    )

    execute_sql(
        engine,
        f"""
        UPDATE "{SCHEMA}"."{SEND_QUEUE_TABLE_NAME}"
        SET
            queue_status = 'pending',
            claim_token = NULL,
            claimed_at = NULL,
            producer_topic = NULL,
            producer_worker_id = NULL
        WHERE queue_status = 'claimed'
          AND producer_sent_at IS NULL
        """
    )

    display(before_reset_dataframe)


---

In [ ]:
# Dispose Engine
engine.dispose()